# ADARIA — estimating ADAR1 p150 / p110 activity from RNA-seq

ADAR1 has two isoforms with different biology: **p110** (nuclear, constitutive) and **p150** (interferon-inducible, largely cytoplasmic — the isoform that edits the double-stranded RNA which MDA5 would otherwise sense as non-self).

Existing summaries (AEI, CEI) return **one number per sample**, with no isoform resolution and no uncertainty. ADARIA returns:

| output | meaning |
|---|---|
| `a150` | p150 editing activity |
| `a110` | p110 editing activity |
| `se_*` | uncertainty on each |
| `abstain` | `True` when the data cannot separate the isoforms |

Activity is on a calibrated scale: **`a = 1`** means "as active as in the reference rescue experiment" that defined the signature; **`a = 0`** means no catalytic activity (knockout).

> **Run this notebook from the `examples/` directory** after `pip install adaria`.
> The editomes in `examples/data/` are 20 % subsamples of real HeLa libraries — small enough to ship, large enough to reproduce the biology.

## How it works (30 seconds)

1. **Signature** — from a controlled p150-only / p110-only rescue experiment, measure how strongly each locus responds to each isoform → a per-locus fingerprint `(s150, s110)`. Built once and shipped with the package.
2. **Inversion** — for a new sample, count edited/total reads at those loci and solve for the single pair `(a150, a110)` that best explains the genome-wide pattern.

The discriminating signal is the **shape** of editing along the isoform-preference axis: its *height* gives `a150 + a110` (≈ what AEI sees), its *slope* gives `a150 − a110` (what AEI cannot see). See `docs/METHOD.md`.

In [ ]:
import pathlib

import pandas as pd

from adaria import ADARIA, build_signature_from_table

DATA = pathlib.Path("data")     # bundled example editomes
pd.set_option("display.width", 120)

## Step 1 — What your input looks like

ADARIA consumes an **editome**: per-site edited / unedited read counts with columns `chr, pos, ref_count, edit_count`.

Generate one from a BAM:

```bash
python -m adaria.pileup_sites --plusbam s.bam --minusbam s.bam \
       --sites known_AtoI_sites.bed --out s.sites.tsv --minbq 25
```

Any caller works as long as the column names match.

In [ ]:
editome = pd.read_csv(DATA / "HeLa_WT_IFNb.sites.tsv.gz", sep="\t")
print(f"{len(editome):,} covered sites")
editome.head()

## Step 2 — Load a signature, and **always check it**

`ADARIA.default()` loads the HEK293T signature that ships with the package.

**Never skip `report()`.** An information-poor signature produces *confidently wrong* activities (wild type read as zero, knockout as non-zero) — and uncertainty alone does not catch that, because it is bias, not variance. The structural checks below do.

In [ ]:
iso = ADARIA.default()
_ = iso.signature.report()

| check | meaning | want |
|---|---|---|
| `median editing / b0` | typical editing vs the zero-activity baseline | **> 1** (ideally ≫) |
| `loci with s<0` | loci whose editing sits below baseline | **< 20 %** |
| `median edited reads` | information content per locus | **≥ 30** |
| `collinearity` | how similar the two fingerprints are | informational only |

Verdict `FAIL` → do not use. `WARN` → usable, interpret cautiously.

## Step 3 — Estimate activity for a new sample

In [ ]:
res = iso.estimate(DATA / "HeLa_WT_IFNb.sites.tsv.gz", sample="HeLa_WT_IFNb")
print(res)

**Reading the result**

- `a150`, `a110` — the two activities. Compare *between samples measured with the same signature*.
- `a150 - a110` — the discriminating contrast; the statistically hardest quantity, so it carries its own SE.
- `n_anchors` — how many signature loci were actually covered. A large drop versus other samples means less information.
- `abstain` — `True` when the contrast is not identifiable; report an abstention rather than a number.

> ⚠️ The reported SEs come from a Laplace approximation with a binomial likelihood and a signature treated as exact. Simulation shows realistic error is **≈ 8× larger**, so use them for *relative* comparison and calibrate any abstention threshold accordingly.

Access values with `res.a150`, `res.se_contrast`, or `res.to_dict()`.

## Step 4 — Several samples at once

In [ ]:
samples = {
    "WT_mock":      DATA / "HeLa_WT_mock.sites.tsv.gz",
    "WT_IFNb":      DATA / "HeLa_WT_IFNb.sites.tsv.gz",
    "p150KO_mock":  DATA / "HeLa_p150KO_mock.sites.tsv.gz",
    "ADAR1KO_mock": DATA / "HeLa_ADAR1KO_mock.sites.tsv.gz",
}
df = iso.estimate_many(samples)
df[["sample", "a150", "a110", "se_contrast", "n_anchors", "abstain"]].round(4)

These four are also the method's built-in sanity check — the estimates should reproduce the known genetics:

| sample | expectation |
|---|---|
| WT mock | both activities positive |
| WT + IFN-β | **`a150` rises** (p150 is interferon-inducible), `a110` roughly flat |
| p150-KO | **`a150` collapses selectively**, `a110` preserved |
| ADAR1-KO | **both go to 0** |

If your own knockout / inhibitor controls do not behave this way, suspect the editome or the signature before the biology.

In [ ]:
d = df.set_index("sample")
print(f"IFN-b raises a150 by {d.loc['WT_IFNb','a150'] - d.loc['WT_mock','a150']:+.3f}"
      f"   (a110 {d.loc['WT_IFNb','a110'] - d.loc['WT_mock','a110']:+.3f})")

da150 = d.loc["p150KO_mock", "a150"] - d.loc["WT_mock", "a150"]
da110 = d.loc["p150KO_mock", "a110"] - d.loc["WT_mock", "a110"]
print(f"p150-KO: a150 {da150:+.3f} vs a110 {da110:+.3f}"
      f"   -> selectivity {da150 / (da150 + da110):.2f}")

print(f"ADAR1-KO: a150={d.loc['ADAR1KO_mock','a150']:.3f}"
      f"  a110={d.loc['ADAR1KO_mock','a110']:.3f}")

## Step 5 — (Advanced) Use your own signature

Instead of the packaged one, build a signature from your own rescue experiment. The table needs per-cluster coordinates plus **read counts for both arms**: `P150_cov`, `P150_Gs`, `P110_cov`, `P110_Gs`.

```python
sig = build_signature_from_table("my_rescue.tsv", min_edited_reads=30)
sig.report()                      # must be OK/WARN — never use a FAIL signature
iso_custom = ADARIA(sig, signature_id="my_rescue")
print(iso_custom.estimate(DATA / "HeLa_WT_IFNb.sites.tsv.gz"))
```

**`min_edited_reads` is the parameter that matters most.** Information about *which* isoform edited a locus scales with `n·π(1−π)`, so a deeply covered but barely edited locus contributes almost nothing but noise — and filtering on **coverage** does not remove those. Building a signature without this filter is the main way to get a `FAIL` verdict.

> ⚠️ **Activities from different signatures are not comparable.** `a = 1` means "as active as in *that* signature's reference experiment", so the unit changes with the signature. The signature used is recorded in `signature_id` — compare only within one.

## Caveats worth knowing

1. **Panel-defined.** Activity is measured on the loci the signature defines — as AEI is defined on Alu and CEI on 3′UTR inverted Alu. Random loss of loci costs precision, not accuracy; *systematic* loss (e.g. by expression) can shift estimates, so compare samples on a common set of covered anchors.
2. **SEs are optimistic** (~8×, see Step 3).
3. **The signature is a plug-in** — its own estimation error is not propagated into the activity uncertainty.
4. **`b0` is a fixed baseline** (default 0.005). If your signature's editing levels are much lower, lower `b0` *and* raise `min_edited_reads` — `report()` will tell you.

---

**More:** [`docs/METHOD.md`](../docs/METHOD.md) (the model) · [`docs/REPRODUCE.md`](../docs/REPRODUCE.md) (regenerate every paper result).